In [ ]:
import os
import zipfile
import tensorflow as tf

URL = "https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip"
DATA_DIR = "/content/cats_and_dogs_filtered"

# Download the zip into /content (file name only, no path)
zip_path = tf.keras.utils.get_file(
    fname="cats_and_dogs_filtered.zip",
    origin=URL,
    cache_dir="/content",
    cache_subdir="",
    extract=False,
)

# Extract it
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

train_dir = os.path.join(DATA_DIR, "train")
test_dir = os.path.join(DATA_DIR, "validation")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# Split training images into training and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

# Load the separate test images
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="binary",
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
# Load MobileNetV2 with pretrained ImageNet weights
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Keep the pretrained weights unchanged
base_model.trainable = False

print("Total base-model layers:", len(base_model.layers))
print("Base model trainable:", base_model.trainable)

In [ ]:

# Build the classifier on top of MobileNetV2
inputs = tf.keras.Input(shape=(224, 224, 3))

# Prepare images for MobileNetV2
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

# Extract features using the frozen base model
x = base_model(x, training=False)

# Add the classification head
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

In [ ]:

# Configure how the model learns
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:


# Stop training if validation loss stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    start_from_epoch=9
)

# Train the classification head
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stopping]
)


In [ ]:

# Evaluate the trained model on the test dataset
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")


In [ ]:
import matplotlib.pyplot as plt

# Select three cats and two dogs from the test set
test_batches = list(test_ds)

cat_images, cat_labels = test_batches[0]
dog_images, dog_labels = test_batches[-1]

images = tf.concat([cat_images[:3], dog_images[:2]], axis=0)
labels = tf.concat([cat_labels[:3], dog_labels[:2]], axis=0)

probabilities = model.predict(images, verbose=0).flatten()

plt.figure(figsize=(15, 5))

for i in range(5):
    dog_probability = float(probabilities[i])
    predicted_index = 1 if dog_probability >= 0.5 else 0
    confidence = (
        dog_probability if predicted_index == 1
        else 1 - dog_probability
    )
    true_index = int(labels[i].numpy()[0])

    plt.subplot(1, 5, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(
        f"Predicted: {class_names[predicted_index]}\n"
        f"Confidence: {confidence:.1%}\n"
        f"Actual: {class_names[true_index]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()